# Extract Feature from Each Frame

In [69]:
# Import libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader

In [104]:
# extract features from the CSV file
def extract_frame_features(frame_df):
    num_objects = len(frame_df)
    avg_conf = frame_df["confidence"].mean()
    avg_box_area = ((frame_df["x_max"] - frame_df["x_min"]) *
                    (frame_df["y_max"] - frame_df["y_min"])).mean()

    desired_size = 13  # fixed number of object classes expected
    class_counts = np.zeros(desired_size)

    for cls in frame_df["object_class"]:
        cls = int(cls)
        if cls < desired_size:
            class_counts[cls] += 1  # only count classes within desired_size

    return torch.tensor([num_objects, avg_conf, avg_box_area] + class_counts.tolist(), dtype=torch.float32)

# Process the CSV file and extract features for each frame
def process_video_csv(csv_path):
    df = pd.read_csv(csv_path)
    sequences = []
    for _, frame_df in df.groupby("frame"):
        frame_feat = extract_frame_features(frame_df)
        sequences.append(frame_feat)
    return torch.stack(sequences) 



# Load Data Labels

In [92]:
import csv

# Load label CSV into a dictionary
label_dict = {}
with open("../data/train_labels.csv", newline = "") as f:
    reader = csv.DictReader(f)
    for row in reader:
        label_dict[row["id"]] = int(row["target"])

# Split Data into Training, Validation, and Test Sets

In [93]:
from sklearn.model_selection import train_test_split

# Get all video directories
video_dirs = sorted([
    os.path.join("../data/yolo_processed_data", d)
    for d in os.listdir("../data/yolo_processed_data")
    if os.path.isdir(os.path.join("../data/yolo_processed_data", d))
])

# Initialize an empty list to store video_ids
video_ids = []

# Extract video IDs from the directory names, removing video_ prefix
for video_dir in video_dirs:
    video_id = os.path.basename(video_dir).replace("video_", "")
    video_ids.append(video_id)

# Ensure there are labels for all video IDs
for video_id in video_ids:
    if video_id not in label_dict:
        raise AssertionError(f"Label missing for video: {video_id}")



# Split the video IDs
train_ids, temp_ids = train_test_split(video_ids, test_size = 0.2)
val_ids, test_ids = train_test_split(temp_ids, test_size = 0.5)


# Initialize empty lists to store the video directories
train_videos = []
val_videos = []
test_videos = []

# Populate the train_videos list with the corresponding directories
for video_id in train_ids:
    video_dir = os.path.join("../data/yolo_processed_data", "video_" + video_id)
    train_videos.append(video_dir)

# Populate the val_videos list with the corresponding directories
for video_id in val_ids:
    video_dir = os.path.join("../data/yolo_processed_data", "video_" + video_id)
    val_videos.append(video_dir)

# Populate the test_videos list with the corresponding directories
for video_id in test_ids:
    video_dir = os.path.join("../data/yolo_processed_data", "video_" + video_id)
    test_videos.append(video_dir)

In [94]:
# Shape Sanity Check
print(f"-----Dataset Sizes-----")
print(f"Train videos: {len(train_videos) / len(video_dirs)}")
print(f"Validation videos: {len(val_videos) / len(video_dirs)}")
print(f"Test videos: {len(test_videos) / len(video_dirs)}\n")


print(f"-----Dataset Values-----")
print(f"Train Videos: {train_videos[0:2]}")
print(f"Validation Videos: {val_videos[0:2]}")
print(f"Test Videos: {test_videos[0:2]}")

-----Dataset Sizes-----
Train videos: 0.8
Validation videos: 0.1
Test videos: 0.1

-----Dataset Values-----
Train Videos: ['../data/yolo_processed_data\\video_01933', '../data/yolo_processed_data\\video_01657']
Validation Videos: ['../data/yolo_processed_data\\video_01846', '../data/yolo_processed_data\\video_00275']
Test Videos: ['../data/yolo_processed_data\\video_00888', '../data/yolo_processed_data\\video_01066']


# Create Dataset and DataLoader

In [95]:
# Create a dataset class for the video sequences
class VideoDataset(Dataset):
    def __init__(self, video_dirs, label_dict):
        self.video_dirs = video_dirs
        self.label_dict = label_dict

    def __len__(self):
        return len(self.video_dirs)
    
    def __getitem__(self, idx):
        video_dir = self.video_dirs[idx]
        video_id = os.path.basename(video_dir)
        label = self.label_dict.get(video_id, 0)

        csv_path = os.path.join(video_dir, "detections.csv")
        seq = process_video_csv(csv_path)

        return seq, torch.tensor(label, dtype = torch.float32)
    
    
# Collate function to pad sequences and create batches
def collate_fn(batch):
    sequences, labels = zip(*batch)
    lengths = [len(seq) for seq in sequences]  # Get the sequence lengths
    padded = pad_sequence(sequences, batch_first = True, padding_value = 0)  # Pad sequences with zero padding
    
    return padded, torch.tensor(lengths), torch.tensor(labels)


# Create datasets for training and validation
train_dataset = VideoDataset(train_videos, label_dict)
val_dataset = VideoDataset(val_videos, label_dict)
test_dataset = VideoDataset(test_videos, label_dict)

# Create DataLoader objects for training, validation, and testing
train_loader = DataLoader(train_dataset, batch_size = 4, shuffle = True, collate_fn = collate_fn)
val_loader = DataLoader(val_dataset, batch_size = 4, shuffle = False, collate_fn = collate_fn)
test_loader = DataLoader(test_dataset, batch_size = 4, shuffle = False, collate_fn = collate_fn)

# Define the LSTM Model

In [96]:
# Define the LSTM model for classification
class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers = 1):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first = True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first = True, enforce_sorted = False)
        _, (hn, _) = self.lstm(packed)
        out = self.fc(hn[-1])
        
        return self.sigmoid(out).squeeze()

# Training Loop

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_dim = 16  # 3 summary features + 10 class counts
model = LSTMClassifier(input_dim, hidden_dim = 64).to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)


for epoch in range(10):
    model.train()
    for x_batch, lengths, y_batch in train_loader:
        x_batch, lengths, y_batch = x_batch.to(device), lengths.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(x_batch, lengths)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")


Epoch 1, Loss: 0.8699
Epoch 2, Loss: 0.8384
Epoch 3, Loss: 0.7943
Epoch 4, Loss: 0.7770
Epoch 5, Loss: 0.7492
Epoch 6, Loss: 0.7198
Epoch 7, Loss: 0.6987
Epoch 8, Loss: 0.6065
Epoch 9, Loss: 0.5867
Epoch 10, Loss: 0.5656
